# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hariommishra-12/Flyrank-Project/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git repo

Cloning into 'repo'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.90 MiB/s, done.
Resolving deltas: 100% (153/153), done.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("repo/data/raw/content_refresh_anonymized.csv")

# Build the proxy label exactly as the starter pipeline defines it.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print(f"declining-label rate: {df['is_declining_label'].mean()*100:.1f}%")
print()
print("trend_direction breakdown (the rule this label is built from):")
print(df["trend_direction"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
declining-label rate: 54.2%

trend_direction breakdown (the rule this label is built from):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The starter pipeline already computed this metric on this exact label + data.
# outputs/model_results.json is gitignored (regenerated per-run), so rather than depend on a
# file that may not exist in a fresh clone, I'm reading the same numbers straight out of the
# repo's *committed* report -- outputs/model_report.md -- so this cell stays reproducible.

with open("repo/outputs/model_report.md") as f:
    report_text = f.read()

# Print just the model comparison table section of the committed report.
start = report_text.index("## Model Comparison")
end = report_text.index("## Final Queue")
print(report_text[start:end].strip())

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis as an actual dataframe: one row per page,
# with the columns a reviewer would actually want to see, plus the proxy target.

cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "days_since_last_update", "word_count",
    "trend_direction", "is_declining_label",
]
print("shape (rows=pages, cols=fields shown):", df[cols].shape)
df[cols].head(5)

shape (rows=pages, cols=fields shown): (30000, 10)


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,20,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,25,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,20,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,22,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,14,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The baseline-vs-model gap that motivates "ML earns its place here",
# read directly from this repo's committed starter results.

print("Model              ROC AUC   Avg precision   Precision@50")
rows = [
    ("baseline_rules",       0.627, 0.468, 0.240),
    ("logistic_regression",  0.700, 0.522, 0.400),
    ("decision_tree",        0.742, 0.575, 0.540),
    ("random_forest",        0.750, 0.618, 0.740),
]
for name, auc, ap, p50 in rows:
    print(f"{name:18s} {auc:6.3f}    {ap:6.3f}          {p50:6.3f}")

print()
print("baseline top-50 correct:", round(0.240 * 50), "of 50")
print("random forest top-50 correct:", round(0.740 * 50), "of 50")

Model              ROC AUC   Avg precision   Precision@50
baseline_rules      0.627     0.468           0.240
logistic_regression  0.700     0.522           0.400
decision_tree       0.742     0.575           0.540
random_forest       0.750     0.618           0.740

baseline top-50 correct: 12 of 50
random forest top-50 correct: 37 of 50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.